In [34]:
import re
import time
import sqlite3
import datetime as dt
from pathlib import Path
import requests
from bs4 import BeautifulSoup
from edgar import set_identity, Company

In [35]:
TICKERS = ['MSFT', 'GOOGL', 'META', 'AMZN', 'NVDA', 'AMD',
           'INTC', 'CRM', 'ORCL', 'ADBE', 'CSCO', 'IBM', 'NOW', 'NFLX']
TRANSCRIPTS_DIR = Path('../data/transcripts')
DB_PATH = '../data/market.db'
set_identity('EarningsCallAnalyst/0.1 (educational use; contact: your-email@example.com)')
SEARCH_DAYS_AFTER = 3
SEARCH_DAYS_BEFORE = 1
DELAY_SECONDS = 0.5
TIMEOUT_SECONDS = 30
SEC_USER_AGENT = 'EarningsCallAnalyst/0.1 (educational use; contact: your-email@example.com)'
TRANSCRIPTS_DIR.mkdir(parents=True, exist_ok=True)

[00:28:02] INFO     Identity of the Edgar REST client set to [EarningsCallAnalyst/0.1 (educational use; core.py:158
                    contact: your-email@example.com)]                                                           

In [36]:
_FILINGS_CACHE = {}
def _get_8k_filings(ticker: str):
    """Get all 8-K filings for a ticker, using a session cache."""
    if ticker not in _FILINGS_CACHE:
        company = Company(ticker)
        _FILINGS_CACHE[ticker] = list(company.get_filings(form='8-K'))
    return _FILINGS_CACHE[ticker]
def _normalize_date(d):
    """Convert a filing_date to a datetime.date object."""
    if isinstance(d, str):
        return dt.datetime.strptime(d[:10], '%Y-%m-%d').date()
    if hasattr(d, 'date'):
        return d.date()
    return d
def find_8k_exhibit(ticker: str, earnings_date: dt.date):
    """
    Find the 8-K filing for a ticker near the given earnings date.
    Returns (exhibit_url, filing_date, exhibit_description) or (None, None, None).
    """
    filings = _get_8k_filings(ticker)
    window_start = earnings_date - dt.timedelta(days=SEARCH_DAYS_BEFORE)
    window_end = earnings_date + dt.timedelta(days=SEARCH_DAYS_AFTER)
    for f in filings:
        fd = _normalize_date(f.filing_date)
        if fd > window_end:
            continue
        if fd < window_start:
            break
        ex_list = list(f.exhibits)
        for ex in ex_list:
            desc = (ex.description or '').upper()
            dtype = (ex.document_type or '').upper()
            if '99' in desc or 'EX-99' in dtype:
                return ex.url, fd, ex.description
        if ex_list:
            return ex_list[0].url, fd, ex_list[0].description
    return None, None, None
def _extract_text_from_html(html: str) -> str:
    """Extract readable text from an SEC filing HTML page."""
    soup = BeautifulSoup(html, 'html.parser')
    for tag in soup(['script', 'style', 'nav', 'footer', 'header']):
        tag.decompose()
    for sel in ['div.body', '.body', 'div[class*="body"]',
                'div[class*="content"]', 'div[class*="text"]']:
        container = soup.select_one(sel)
        if container:
            text = container.get_text(' ', strip=True)
            if len(text) > 200:
                return re.sub(r'\s+', ' ', text).strip()
    body = soup.find('body') or soup
    text = body.get_text(' ', strip=True)
    return re.sub(r'\s+', ' ', text).strip()
def scrape_transcript(ticker: str, quarter: str, year: int,
                      earnings_date: dt.date,
                      overwrite: bool = False) -> dict | None:
    """Fetch one transcript from SEC EDGAR. Returns a metadata dict, or None if skipped/failed.
    Skips silently if the local file already exists and overwrite=False.
    """
    outdir = TRANSCRIPTS_DIR / ticker
    outdir.mkdir(parents=True, exist_ok=True)
    outfile = outdir / f'{quarter.lower()}{year}.txt'
    if outfile.exists() and not overwrite:
        return None
    ex_url, filing_date, ex_desc = find_8k_exhibit(ticker, earnings_date)
    if ex_url is None:
        return {
            'ticker': ticker, 'quarter': quarter.lower(), 'year': year,
            'pub_date': earnings_date.isoformat(), 'url': 'NOT_FOUND',
            'file_path': str(outfile), 'word_count': 0,
            'status': 404, 'attempt': 1,
            'scrape_time': dt.datetime.utcnow().isoformat(timespec='seconds'),
        }
    headers = {'User-Agent': SEC_USER_AGENT, 'Accept': 'text/html,application/xhtml+xml'}
    last_err = None
    for attempt in range(2):
        try:
            r = requests.get(ex_url, headers=headers, timeout=TIMEOUT_SECONDS)
            if r.status_code == 200:
                text = _extract_text_from_html(r.text)
                outfile.write_text(text, encoding='utf-8')
                return {
                    'ticker': ticker,
                    'quarter': quarter.lower(),
                    'year': year,
                    'pub_date': earnings_date.isoformat(),
                    'url': ex_url,
                    'file_path': str(outfile),
                    'word_count': len(text.split()),
                    'status': 200,
                    'attempt': attempt + 1,
                    'scrape_time': dt.datetime.utcnow().isoformat(timespec='seconds'),
                }
            elif 500 <= r.status_code < 600 and attempt == 0:
                time.sleep(DELAY_SECONDS)
                continue
            else:
                return {
                    'ticker': ticker, 'quarter': quarter.lower(), 'year': year,
                    'pub_date': earnings_date.isoformat(), 'url': ex_url,
                    'file_path': str(outfile), 'word_count': 0,
                    'status': r.status_code, 'attempt': attempt + 1,
                    'scrape_time': dt.datetime.utcnow().isoformat(timespec='seconds'),
                }
        except requests.RequestException as e:
            last_err = e
            if attempt == 0:
                time.sleep(DELAY_SECONDS)
                continue
    print(f"[{ticker}/{quarter.lower()}{year}] network error after retries: {last_err}")
    return None

In [37]:
TEST_TARGETS = [
    ('MSFT', 'q3', 2024, dt.date(2024, 4, 25)),
    ('MSFT', 'q4', 2024, dt.date(2024, 7, 30)),
]
results = []
for ticker, quarter, year, earnings_date in TEST_TARGETS:
    res = scrape_transcript(ticker, quarter, year, earnings_date)
    if res is None:
        print(f"  {ticker} {quarter.upper()}{year}: skipped (already on disk)")
        continue
    results.append(res)
    if res['status'] == 200:
        flag = '\u26a0 tiny body' if res['word_count'] < 200 else '\u2713 OK'
    elif res['status'] == 404 and res['url'] == 'NOT_FOUND':
        flag = 'NO 8-K FOUND'
    else:
        flag = f"HTTP {res['status']}"
    print(f"  {ticker} {quarter.upper()}{year}: {flag} — "
          f"{res['word_count']:,} words -> {res['file_path']}")
    if res['status'] != 200:
        print(f"     url: {res['url']}")
    time.sleep(DELAY_SECONDS)
print(f"\nScraped {len(results)} new transcripts (out of {len(TEST_TARGETS)} attempted)")

  MSFT Q32024: skipped (already on disk)
  MSFT Q42024: skipped (already on disk)

Scraped 0 new transcripts (out of 2 attempted)


In [38]:
print("=== Files in data/transcripts/MSFT/ ===")
msft_dir = TRANSCRIPTS_DIR / 'MSFT'
if msft_dir.exists():
    for p in sorted(msft_dir.glob('*.txt')):
        print(f"  {p.name:>10s}  {p.stat().st_size:>8,} bytes")
else:
    print("  (directory does not yet exist)")
print("\n=== First 400 chars of earliest scrape ===")
txt_files = sorted(msft_dir.glob('*.txt')) if msft_dir.exists() else []
if txt_files:
    print(txt_files[0].read_text(encoding='utf-8')[:400])

=== Files in data/transcripts/MSFT/ ===
  q12026.txt    58,920 bytes
  q22026.txt    60,518 bytes
  q32024.txt    17,708 bytes
  q42024.txt    21,529 bytes

=== First 400 chars of earliest scrape ===
Image source: The Motley Fool. Date Oct. 29, 2025 at 5:30 p.m. ET Call participants Chairman and Chief Executive Officer — Satya Nadella Chief Financial Officer — Amy E. Hood Chief Accounting Officer — Alice Jolla Investor Relations — Jonathan Neilson Legal — Keith Dolliver Need a quote from a Motley Fool analyst? Email [email protected] Risks Azure capacity constraints — Amy E. Hood stated, "We n


In [40]:
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()
cur.execute('''
CREATE TABLE IF NOT EXISTS transcripts (
    ticker      TEXT,
    quarter     TEXT,
    year        INTEGER,
    pub_date    DATE,
    url         TEXT,
    file_path   TEXT,
    word_count  INTEGER,
    status      INTEGER,
    attempt     INTEGER,
    scrape_time TEXT,
    PRIMARY KEY (ticker, quarter, year)
)
''')
for res in results:
    cur.execute('''
        INSERT OR REPLACE INTO transcripts
          (ticker, quarter, year, pub_date, url, file_path,
           word_count, status, attempt, scrape_time)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    ''', (res['ticker'], res['quarter'], res['year'], res['pub_date'],
          res['url'], res['file_path'], res['word_count'],
          res['status'], res['attempt'], res['scrape_time']))
conn.commit()
conn.close()
print(f"Logged {len(results)} rows to 'transcripts' table")

Logged 0 rows to 'transcripts' table


In [41]:
_CY_RULES = [(1, 3, 'q4', -1), (4, 6, 'q1', 0), (7, 9, 'q2', 0), (10, 12, 'q3', 0)]
FISCAL_RULES = {
    'MSFT': [(7, 9, 'q4', 0), (10, 12, 'q1', 1), (1, 3, 'q2', 0), (4, 6, 'q3', 0)],
    'NVDA': [(2, 4, 'q4', 0), (5, 7, 'q1', 1), (8, 10, 'q2', 1), (11, 1, 'q3', 1)],
    'CRM':  [(2, 4, 'q4', 0), (5, 7, 'q1', 1), (8, 10, 'q2', 1), (11, 1, 'q3', 1)],
    'ORCL': [(6, 8, 'q4', 0), (9, 11, 'q1', 1), (12, 12, 'q2', 1),
             (1, 1, 'q2', 0), (2, 5, 'q3', 0)],
    'ADBE': [(12, 12, 'q4', 0), (1, 1, 'q4', -1), (2, 4, 'q1', 0),
             (5, 7, 'q2', 0), (8, 11, 'q3', 0)],
    'CSCO': [(7, 9, 'q4', 0), (10, 12, 'q1', 1), (1, 3, 'q2', 0), (4, 6, 'q3', 0)],
}
for _t in ('GOOGL', 'META', 'AMZN', 'IBM', 'NOW', 'NFLX', 'AMD', 'INTC'):
    if _t not in FISCAL_RULES:
        FISCAL_RULES[_t] = _CY_RULES
def get_fiscal_quarter_year(ticker, announcement_date):
    """Map an earnings announcement date → (fiscal_quarter_label, fiscal_year)."""
    rules = FISCAL_RULES.get(ticker)
    if not rules:
        raise ValueError(f"No FISCAL_RULES entry for {ticker}")
    month = announcement_date.month
    year  = announcement_date.year
    for m_start, m_end, q_label, offset in rules:
        if m_start <= m_end:
            if m_start <= month <= m_end:
                return (q_label, year + offset)
        else:
            if month >= m_start or month <= m_end:
                return (q_label, year + offset)
    raise ValueError(f"{ticker} announcement month {month} not in FISCAL_RULES")
conn = sqlite3.connect(DB_PATH)
earnings_rows = conn.execute("""
    SELECT ticker, earnings_date
    FROM earnings
    WHERE earnings_date <= date('now')
      AND reported_eps IS NOT NULL        -- only historical (reported) quarters
    ORDER BY ticker, earnings_date
""").fetchall()
conn.close()
SCALE_TARGETS = []
fiscal_errors = []
for ticker, earnings_date_str in earnings_rows:
    edate = dt.datetime.strptime(earnings_date_str.split(' ')[0], '%Y-%m-%d').date()
    try:
        quarter, fy = get_fiscal_quarter_year(ticker, edate)
        SCALE_TARGETS.append((ticker, quarter, fy, edate))
    except ValueError as e:
        fiscal_errors.append((ticker, str(edate), str(e)))
if fiscal_errors:
    print(f'WARNING: {len(fiscal_errors)} rows could not be mapped:')
    for t, d, err in fiscal_errors[:8]:
        print(f'  {t} {d}: {err}')
print(f'Built {len(SCALE_TARGETS)} scale targets from earnings table'
      f' ({len(fiscal_errors)} fiscal-mapping gaps)')
print(f'Estimated runtime: {len(SCALE_TARGETS) * DELAY_SECONDS / 60:.0f} min '
      f'at {DELAY_SECONDS:.0f}s delay')
print(f'First 5 targets:')
for t in SCALE_TARGETS[:5]:
    print(f'  {t[0]:6s} {t[1]:3s} FY{t[2]}  earnings_date={t[3]}')
print(f'... ({len(SCALE_TARGETS)} total)')
results = []
successes = 0
failures = 0
skips = 0
print(f'\n=== Starting scale fetch ({len(SCALE_TARGETS)} targets) ===')
t_start = time.time()
for i, (ticker, quarter, year, earnings_date) in enumerate(SCALE_TARGETS):
    try:
        res = scrape_transcript(ticker, quarter, year, earnings_date)
    except Exception as e:
        failures += 1
        if (i + 1) % 10 == 0:
            elapsed = (time.time() - t_start) / 60
            print(f'  [{i+1}/{len(SCALE_TARGETS)}] {ticker} {quarter}{year}: ERROR \u2014 {e}')
        if i < len(SCALE_TARGETS) - 1:
            time.sleep(DELAY_SECONDS)
        continue
    if res is None:
        skips += 1
        if (i + 1) % 30 == 0:
            elapsed = (time.time() - t_start) / 60
            print(f'  [{i+1}/{len(SCALE_TARGETS)}] \u2026 {skips} skips so far '
                  f'({elapsed:.0f} min elapsed)')
        if i < len(SCALE_TARGETS) - 1:
            time.sleep(DELAY_SECONDS)
        continue
    results.append(res)
    if res['status'] == 200:
        successes += 1
        flag = '\u26a0 tiny body' if res['word_count'] < 200 else '\u2713'
    elif res['status'] == 404 and res['url'] == 'NOT_FOUND':
        failures += 1
        flag = 'NO 8-K'
    else:
        failures += 1
        flag = f"HTTP {res['status']}"
    if (i + 1) % 10 == 0 or res['status'] != 200:
        elapsed = (time.time() - t_start) / 60
        print(f'  [{i+1}/{len(SCALE_TARGETS)}] {ticker} {quarter}{year}: {flag} \u2014 '
              f'{res["word_count"]:,} words  ({elapsed:.0f} min)')
        if res['status'] != 200 and res['url'] != 'NOT_FOUND':
            print(f'     url: {res["url"]}')
    if i < len(SCALE_TARGETS) - 1:
        time.sleep(DELAY_SECONDS)
elapsed = (time.time() - t_start) / 60
print(f'\nDone in {elapsed:.0f} min: {successes} ok, {failures} failed, '
      f'{skips} skipped (out of {len(SCALE_TARGETS)} total)')
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()
cur.execute('''
    CREATE TABLE IF NOT EXISTS transcripts (
        ticker      TEXT,
        quarter     TEXT,
        year        INTEGER,
        pub_date    DATE,
        url         TEXT,
        file_path   TEXT,
        word_count  INTEGER,
        status      INTEGER,
        attempt     INTEGER,
        scrape_time TEXT,
        PRIMARY KEY (ticker, quarter, year)
    )
''')
for res in results:
    cur.execute('''
        INSERT OR REPLACE INTO transcripts
          (ticker, quarter, year, pub_date, url, file_path,
           word_count, status, attempt, scrape_time)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    ''', (res['ticker'], res['quarter'], res['year'], res['pub_date'],
          res['url'], res['file_path'], res['word_count'],
          res['status'], res['attempt'], res['scrape_time']))
conn.commit()
conn.close()
print(f'Logged {len(results)} rows to transcripts table')

Built 420 scale targets from earnings table (0 fiscal-mapping gaps)
Estimated runtime: 4 min at 0s delay
First 5 targets:
  ADBE   q1  FY2019  earnings_date=2019-03-14
  ADBE   q2  FY2019  earnings_date=2019-06-18
  ADBE   q3  FY2019  earnings_date=2019-09-17
  ADBE   q4  FY2019  earnings_date=2019-12-12
  ADBE   q1  FY2020  earnings_date=2020-03-12
... (420 total)

=== Starting scale fetch (420 targets) ===
  [10/420] ADBE q22021: ✓ — 2,332 words  (0 min)
  [20/420] ADBE q42023: ✓ — 3,150 words  (1 min)
  [30/420] ADBE q22026: ✓ — 3,078 words  (1 min)
  [40/420] AMD q12021: ✓ — 4,387 words  (1 min)
  [50/420] AMD q32023: ✓ — 4,880 words  (1 min)
  [60/420] AMD q12026: ✓ — 4,290 words  (1 min)
  [70/420] AMZN q12021: ✓ — 9,486 words  (2 min)
  [90/420] AMZN q12026: ✓ — 5,847 words  (2 min)
  [100/420] CRM q12022: ✓ — 5,410 words  (3 min)
  [110/420] CRM q32024: ✓ — 6,443 words  (3 min)
  [120/420] CRM q12027: ✓ — 6,466 words  (3 min)
  [130/420] CSCO q32021: ✓ — 5,062 words  (3 min)
  